In [ ]:
# # install octave
# !sudo apt-get -qq update
# !sudo apt-get -qq install octave octave-signal liboctave-dev

# # install oct2py that compatible with colab
# import os

# from pkg_resources import get_distribution

# os.system(
#     f"pip install -qq"
#     f" ipykernel=={get_distribution('ipykernel').version}"
#     f" ipython=={get_distribution('ipython').version}"
#     f" tornado=={get_distribution('tornado').version}"
#     f" oct2py"
# )

# # install packages
# !pip install -qq matpower matpowercaseframesa

In [ ]:
import oct2py

import matpower

print(f"oct2py version: {oct2py.__version__}")
print(f"matpower version: {matpower.__version__}")

oct2py version: 5.8.0
matpower version: 8.1.0.2.2.4


In [ ]:
import os

import pandas as pd
from matpowercaseframes import CaseFrames

from matpower import path_matpower, start_instance

In [ ]:
path = os.path.join(path_matpower, "data/case9.m")

In [ ]:
m = start_instance()

## Original Case

In [ ]:
cf = CaseFrames(path, load_case_engine=m)

mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()
cf.branch

,F_BUS,T_BUS,BR_R,BR_X,BR_B,RATE_A,RATE_B,RATE_C,TAP,SHIFT,BR_STATUS,ANGMIN,ANGMAX,PF,QF,PT,QT
branch,,,,,,,,,,,,,,,,,
1,1,4,0.0000,0.0576,0.000,250,250,250,0,0,1,-360,360,71.641021,27.045924,-71.641021,-23.923127
2,4,5,0.0170,0.0920,0.158,250,250,250,0,0,1,-360,360,30.703670,1.030006,-30.537263,-16.543365
3,5,6,0.0390,0.1700,0.358,150,150,150,0,0,1,-360,360,-59.462737,-13.456635,60.816586,-18.074836
4,3,6,0.0000,0.0586,0.000,300,300,300,0,0,1,-360,360,85.000000,-10.859709,-85.000000,14.955327
5,6,7,0.0119,0.1008,0.209,150,150,150,0,0,1,-360,360,24.183414,3.119508,-24.095417,-24.295823
6,7,8,0.0085,0.0720,0.149,250,250,250,0,0,1,-360,360,-75.904583,-10.704177,76.379866,-0.797331
7,8,2,0.0000,0.0625,0.000,250,250,250,0,0,1,-360,360,-163.000000,9.178149,163.000000,6.653660
8,8,9,0.0320,0.1610,0.306,250,250,250,0,0,1,-360,360,86.620134,-8.380817,-84.320163,-11.312751
9,9,4,0.0100,0.0850,0.176,250,250,250,0,0,1,-360,360,-40.679837,-38.687249,40.937352,22.893121


In [ ]:
lines_index = cf.branch.index[cf.branch["BR_R"] > 0]

cf.branch_ext = cf.branch.copy()
cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
cf.branch_ext["SFT_MAX"] = (
    cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
).pow(0.5)
cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]

,RATE_A,SFT_MAX,LOADING
branch,,,
2,250,34.876902,0.139508
3,150,63.445700,0.422971
5,150,34.280089,0.228534
6,250,77.126282,0.308505
8,250,87.355744,0.349423
9,250,56.325571,0.225302


## Derating

In [ ]:
import numpy as np


def get_branch_thermal_correction_factor(
    t_ambient,
    t_max: float = 90.0,
    t_ref: float = 30.0,
) -> float | pd.Series:
    """
    Compute the ambient temperature correction factor (CF) for conductor ratings.

        CF = sqrt((t_max - t_ambient) / (t_max - t_ref))

    Supports scalar inputs (single CF) or per-branch inputs (Series of CFs),
    allowing each branch to have its own ambient condition and insulation class.

    For a peak hot day with heat wave scenario, this is an overestimating the line
    capability (too small derating) since it assumes:
        - ignore solar heating (overestimate)
        - assume cooling the same (overestimate, heat wave often has low wind, thus
        less cooling)
        - ignore resistive heating from current (overestimate)
        - conductor temperature is at max (underestimate, derating too much for lines
        that are not critically loaded, but fair for critically loaded and overloaded
        lines)
        - ignore radiative cooling (underestimate, radiative cooling increases with
        conductor temperature but is ignored in the formula, but this is likely a small
        effect compared to the others)
    where it overestimate is more significant than underestimate (TODO: proof this).

    Parameters
    ----------
    t_ambient : float or pd.Series
        Actual ambient temperature [°C].
        - float   : same temperature applied to all branches
        - Series  : one value per branch (must match branch DataFrame index)

    t_max : float or pd.Series, optional
        Maximum conductor temperature [°C]. Default 90.0 °C (XLPE insulation).
        - float   : same insulation class for all branches
        - Series  : per-branch insulation class

        Common insulation classes (NEC Table 310.15(B)(1)):
            - 60.0 °C : TW, UF
            - 75.0 °C : THW, THWN, XHHW (wet)
            - 90.0 °C : THWN-2, XHHW-2, USE-2  ← default

    t_ref : float or pd.Series, optional
        Reference ambient temperature of the base ratings [°C]. Default 30.0 °C.
        - NEC NFPA 70 / IEC 60364-5-52 : 30.0 °C  ← default
        - IEEE 835                     : 25.0 °C

    Returns
    -------
    float or pd.Series
        CF >= 0, dimensionless.
        - CF = 1.0 : t_ambient == t_ref     (no derating)
        - CF < 1.0 : t_ambient >  t_ref     (derating required)
        - CF > 1.0 : t_ambient <  t_ref     (uprating permitted)
        - CF = 0.0 : t_ambient >= t_max     (line must not carry current)

    Notes
    -----
    - Formula : Neher & McGrath (1957), IEEE Std 738, NEC §310.15
    - t_max   : NEC Table 310.15(B)(1)
    - t_ref   : NEC NFPA 70 / IEC 60364-5-52

    Read more
    ---------
    https://ieeexplore.ieee.org/document/4499653  # Neher & McGrath (1957)
    https://en.wikipedia.org/wiki/Neher%E2%80%93McGrath_method  # Wikipedia
    https://ecalpro.com/ko/standards/nec/table-310-15-b1-temperature  # NEC Table
    https://ieeexplore.ieee.org/document/10382442  # IEEE Std 738
    https://ieeexplore.ieee.org/document/7111195  # IEEE 835
    https://ieeexplore.ieee.org/document/11030252  # IEEE P848/D1
    """
    return np.sqrt(np.maximum(t_max - t_ambient, 0) / (t_max - t_ref))


def apply_derating_to_branch(
    branch: pd.DataFrame,
    t_ambient=40.0,
    t_max=90.0,
    t_ref=30.0,
    rate_columns: list[str] = None,
) -> pd.DataFrame:
    """
    Apply ambient temperature derating to a MatpowerCaseFrames branch DataFrame.

        S_derated = S_rated * CF

    Supports per-branch temperature parameters — each branch can have its own
    ambient temperature, insulation class, and reference temperature.

    Parameters
    ----------
    branch : pd.DataFrame
        cf.branch from a MatpowerCaseFrames object.
        Contains MVA ratings: RATE_A (continuous), RATE_B (emergency),
        RATE_C (emergency peak). See MATPOWER Manual v8.1, Table B-3.
        https://matpower.org/docs/MATPOWER-manual-8.1.pdf

    t_ambient : float or pd.Series, optional
        Ambient temperature [°C]. Default 40.0 °C.
        Pass a Series (aligned to branch.index) for per-branch values.

    t_max : float or pd.Series, optional
        Max conductor temperature [°C]. Default 90.0 °C (XLPE).
        Pass a Series for per-branch insulation classes.

    t_ref : float or pd.Series, optional
        Reference ambient [°C]. Default 30.0 °C (NEC/IEC).
        Pass a Series for per-branch reference temperatures.

    rate_columns : list of str, optional
        Columns to derate. Default: ['RATE_A', 'RATE_B', 'RATE_C'].

    Returns
    -------
    pd.DataFrame
        Copy of branch with derated rating columns and a 'CF' column.

    Examples
    --------
    Scalar — same temperature for all branches:

    >>> branch_derated = apply_derating_to_branch(cf.branch, t_ambient=45.0)

    Per-branch — each branch has its own ambient temperature:

    >>> t = pd.Series([35.0, 42.0, 50.0], index=cf.branch.index)
    >>> branch_derated = apply_derating_to_branch(cf.branch, t_ambient=t)

    Per-branch — different insulation classes too:

    >>> t_max = pd.Series([75.0, 90.0, 90.0], index=cf.branch.index)
    >>> branch_derated = apply_derating_to_branch(cf.branch, t_ambient=t, t_max=t_max)
    """
    if rate_columns is None:
        rate_columns = ["RATE_A", "RATE_B", "RATE_C"]

    correction_factor = get_branch_thermal_correction_factor(
        t_ambient=t_ambient,
        t_max=t_max,
        t_ref=t_ref,
    )

    for col in rate_columns:
        branch[col] = branch[col] * correction_factor

    return branch

In [ ]:
branch_temp = pd.DataFrame(
    {
        "t_ambient": [40.0, 55.0, 45.0, 50.0, 50.0, 38.0],
        "t_max": [90.0, 60.0, 75.0, 90.0, 60.0, 90.0],
        "t_ref": [30.0, 30.0, 30.0, 30.0, 30.0, 30.0],
    },
    index=lines_index,
)

In [ ]:
correction_factors = get_branch_thermal_correction_factor(
    t_ambient=branch_temp["t_ambient"],
    t_max=branch_temp["t_max"],
    t_ref=branch_temp["t_ref"],
)
correction_factors

branch
2    0.912871
3    0.408248
5    0.816497
6    0.816497
8    0.577350
9    0.930949
dtype: float64

In [ ]:
col_rates = ["RATE_A", "RATE_B", "RATE_C"]
cf.branch_ext[col_rates]

,RATE_A,RATE_B,RATE_C
branch,,,
1,250,250,250
2,250,250,250
3,150,150,150
4,300,300,300
5,150,150,150
6,250,250,250
7,250,250,250
8,250,250,250
9,250,250,250


In [ ]:
cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
cf.branch_ext.loc[lines_index, col_rates] = (
    cf.branch_ext.loc[lines_index, col_rates]
    * correction_factors[lines_index].values[:, np.newaxis]
)
cf.branch_ext[col_rates]

,RATE_A,RATE_B,RATE_C
branch,,,
1,250.000000,250.000000,250.000000
2,228.217732,228.217732,228.217732
3,61.237244,61.237244,61.237244
4,300.000000,300.000000,300.000000
5,122.474487,122.474487,122.474487
6,204.124145,204.124145,204.124145
7,250.000000,250.000000,250.000000
8,144.337567,144.337567,144.337567
9,232.737334,232.737334,232.737334


In [ ]:
cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]
print(cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]])

            RATE_A    SFT_MAX   LOADING
branch                                 
2       228.217732  34.876902  0.152823
3        61.237244  63.445700  1.036064
5       122.474487  34.280089  0.279896
6       204.124145  77.126282  0.377840
8       144.337567  87.355744  0.605218
9       232.737334  56.325571  0.242013


## Cascading

In [ ]:
# overloaded lines status update
overloaded = cf.branch_ext.loc[:, "LOADING"] > 1.0
cf.branch.loc[overloaded, "BR_STATUS"] = 0

In [ ]:
mpc = m.runpf(cf.to_mpc(), verbose=False)
cf = CaseFrames(mpc)
cf.infer_numpy()

cf.branch_ext = cf.branch.copy()
cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
cf.branch_ext["SFT_MAX"] = (
    cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
).pow(0.5)

cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
cf.branch_ext.loc[lines_index, col_rates] = (
    cf.branch_ext.loc[lines_index, col_rates]
    * correction_factors[lines_index].values[:, np.newaxis]
)

cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]

,RATE_A,SFT_MAX,LOADING
branch,,,
2,228.217732,96.422435,0.422502
3,61.237244,0.000000,0.000000
5,122.474487,52.150544,0.425807
6,204.124145,69.892165,0.342400
8,144.337567,0.000000,0.000000
9,232.737334,136.832858,0.587928


In [ ]:
# overloaded lines status update
overloaded = cf.branch_ext.loc[:, "LOADING"] > 1.0
cf.branch.loc[overloaded, "BR_STATUS"] = 0

In [ ]:
cf.branch[["F_BUS", "T_BUS", "BR_STATUS"]]

,F_BUS,T_BUS,BR_STATUS
branch,,,
1,1,4,1
2,4,5,1
3,5,6,0
4,3,6,1
5,6,7,1
6,7,8,1
7,8,2,1
8,8,9,0
9,9,4,1


In [ ]:
[groups, isolated] = m.find_islands(cf.to_mpc(), nout=2)
groups, isolated

(Cell([[array([[2., 3., 6., 7., 8.]]), array([[1., 4., 5., 9.]])]],
       dtype=object),
 array([], shape=(1, 0), dtype=float64))

In [ ]:
mpcs = m.extract_islands(mpc, groups)

cfs = {}
for isl, mpc in enumerate(mpcs.flatten()):
    cfs[isl] = CaseFrames(mpc)
    cfs[isl].infer_numpy()
    display(cfs[isl].branch[["F_BUS", "T_BUS", "BR_STATUS"]])

,F_BUS,T_BUS,BR_STATUS
branch,,,
1,3,6,1
2,6,7,1
3,7,8,1
4,8,2,1


,F_BUS,T_BUS,BR_STATUS
branch,,,
1,1,4,1
2,4,5,1
3,9,4,1


In [ ]:
cf = cfs[0]
display(cf.bus)  # we can see, there is no slack bus
if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
    slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
    slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
    cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus
display(cf.bus)

,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,2,2,0,0,0,0,1,1.025000,-116.561496,345,1,1.1,0.9
2,3,2,0,0,0,0,1,1.025000,-118.351989,345,1,1.1,0.9
3,6,1,0,0,0,0,1,1.018568,-121.037729,345,1,1.1,0.9
4,7,1,100,35,0,0,1,1.003676,-123.723469,345,1,1.1,0.9
5,8,1,0,0,0,0,1,1.017025,-121.037729,345,1,1.1,0.9


,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,2,2,0,0,0,0,1,1.025000,-116.561496,345,1,1.1,0.9
2,3,3,0,0,0,0,1,1.025000,-118.351989,345,1,1.1,0.9
3,6,1,0,0,0,0,1,1.018568,-121.037729,345,1,1.1,0.9
4,7,1,100,35,0,0,1,1.003676,-123.723469,345,1,1.1,0.9
5,8,1,0,0,0,0,1,1.017025,-121.037729,345,1,1.1,0.9


In [ ]:
cf = cfs[1]
cf.bus  # we can see, there is slack bus

,BUS_I,BUS_TYPE,PD,QD,GS,BS,BUS_AREA,VM,VA,BASE_KV,ZONE,VMAX,VMIN
bus,,,,,,,,,,,,,
1,1,3,0,0,0,0,1,1.040000,0.000000,345,1,1.1,0.9
2,4,1,0,0,0,0,1,0.988486,-7.036933,345,1,1.1,0.9
3,5,1,90,30,0,0,1,0.946520,-11.874571,345,1,1.1,0.9
4,9,1,125,50,0,0,1,0.930215,-13.406540,345,1,1.1,0.9


In [ ]:
for _isl, cf in cfs.items():
    if (cf.bus["BUS_TYPE"] == 3).sum() == 0:
        slack_gen_idx = cf.gen.index[cf.gen["PMAX"].argmax()]
        slack_bus_idx = cf.gen.loc[slack_gen_idx, "GEN_BUS"]
        cf.bus.loc[slack_bus_idx, "BUS_TYPE"] = 3  # set slack bus

    m.runpf(cf.to_mpc(), verbose=False)
    mpc = m.runpf(cf.to_mpc(), verbose=False)
    cf = CaseFrames(mpc)
    cf.infer_numpy()

    cf.branch_ext = cf.branch.copy()
    cf.branch_ext["PFT_MAX"] = cf.branch_ext[["PF", "PT"]].abs().max(axis=1)
    cf.branch_ext["QFT_MAX"] = cf.branch_ext[["QF", "QT"]].abs().max(axis=1)
    cf.branch_ext["SFT_MAX"] = (
        cf.branch_ext[["PFT_MAX", "QFT_MAX"]].pow(2).sum(axis=1)
    ).pow(0.5)

    lines_index = cf.branch.index[cf.branch["BR_R"] > 0]

    cf.branch_ext[col_rates] = cf.branch_ext[col_rates].astype(float)
    cf.branch_ext.loc[lines_index, col_rates] = (
        cf.branch_ext.loc[lines_index, col_rates]
        * correction_factors[lines_index].values[:, np.newaxis]
    )

    cf.branch_ext["LOADING"] = cf.branch_ext["SFT_MAX"] / cf.branch_ext["RATE_A"]
    display(
        cf.branch_ext.loc[lines_index, ["RATE_A", "SFT_MAX", "LOADING"]]
    )  # see, we can further cascade

,RATE_A,SFT_MAX,LOADING
branch,,,
2,136.930639,68.754205,0.502110
3,102.062073,163.126766,1.598309


,RATE_A,SFT_MAX,LOADING
branch,,,
2,228.217732,96.422435,0.422502
3,102.062073,136.832858,1.340683
